# 🗂️ Bloque 1 — Sesión 1: Naturaleza y tipos de datos

**Módulo:** Depuración, limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos  
**Duración estimada:** ~4 horas.

---

### Contenidos de esta sesión
- **1.1** Naturaleza de los datos: estructurados, semiestructurados y no estructurados
- **1.2** Tipos de variables: numéricas, categóricas, temporales y booleanas

### Criterio de evaluación asociado
> **(CE a)** Se han descrito técnicas y posibilidades de almacenaje de los datos para componer conjuntos, identificando su naturaleza.

In [ ]:
# Instalaciones y librerías necesarias (ejecutar una sola vez)
import pandas as pd
import json
import seaborn as sns

print('✅ Librerías cargadas correctamente')

---
## 1.1 Naturaleza de los datos

Los datos que usaremos para entrenar modelos de Machine Learning pueden tener tres naturalezas distintas según su organización:

| Tipo | Descripción | Ejemplos |
|------|-------------|----------|
| **Estructurados** | Organizados en filas y columnas. Fácilmente tabulables. | Tablas SQL, CSV, Excel |
| **Semiestructurados** | Tienen alguna organización pero no son tabulares. | JSON, XML, HTML |
| **No estructurados** | Sin organización predefinida. | Texto libre, imágenes, audio, vídeo |

> 💡 En este módulo trabajaremos principalmente con datos **estructurados** y **semiestructurados**, que son los más habituales en proyectos de ML supervisado.

### 📌 Ejemplo 1 — Datos estructurados con pandas

In [ ]:
# Los datos estructurados se representan fácilmente como un DataFrame
data = {
    'nombre':       ['Ana', 'Luis', 'María', 'Carlos', 'Elena'],
    'edad':         [23, 35, 28, 42, 31],
    'salario':      [28000.0, 45000.0, 32000.0, 55000.0, 38000.0],
    'departamento': ['IT', 'Ventas', 'IT', 'RRHH', 'Ventas'],
    'activo':       [True, True, False, True, True]
}

df = pd.DataFrame(data)
print('Datos estructurados — DataFrame de pandas:')
print(df)
print(f'\nDimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

### 📌 Ejemplo 2 — Datos semiestructurados (JSON)

In [ ]:
# JSON: tiene estructura, pero no es directamente tabular
# Fíjate en que cada empleado puede tener un número distinto de habilidades
json_data = '''
{
  "empleados": [
    {"nombre": "Ana",    "edad": 23, "habilidades": ["Python", "SQL"]},
    {"nombre": "Luis",   "edad": 35, "habilidades": ["Excel", "Power BI", "SQL"]},
    {"nombre": "María",  "edad": 28, "habilidades": ["Python", "R"]}
  ]
}
'''

datos = json.loads(json_data)
print('Datos semiestructurados — JSON:')
for emp in datos['empleados']:
    print(f"  {emp['nombre']} ({emp['edad']} años): {', '.join(emp['habilidades'])}")

# Intentamos convertirlo a DataFrame
print('\nConvertido a DataFrame con pd.json_normalize():')
df_json = pd.json_normalize(datos['empleados'])
print(df_json)
print('\n⚠️ La columna "habilidades" contiene listas → estructura irregular')

### 💡 ¿Cómo entrenar un modelo con datos semiestructurados?

Los modelos de ML necesitan datos tabulares (filas × columnas de **valores escalares**). Cuando los datos semiestructurados contienen listas de longitud variable —como las habilidades del ejemplo anterior— hay que **aplanarlos** antes de poder entrenar cualquier modelo.

La estrategia más habitual es el **multi-hot encoding**: crear una columna binaria por cada valor posible de la lista.

```
JSON / XML  (semiestructurado)
        ↓   pd.json_normalize() + transformaciones
DataFrame tabular  (estructurado)
        ↓   encoding, escalado...
Modelo de ML
```

### 📌 Ejemplo 2b — Aplanar datos semiestructurados con multi-hot encoding

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Partimos del df_json del Ejemplo 2
mlb = MultiLabelBinarizer()
habilidades_encoded = mlb.fit_transform(df_json['habilidades'])

df_modelo = pd.concat([
    df_json[['nombre', 'edad']],
    pd.DataFrame(habilidades_encoded, columns=mlb.classes_)
], axis=1)

print('DataFrame resultante tras el multi-hot encoding:')
print(df_modelo)
print('\nCada habilidad es ahora una columna binaria (1 = tiene la habilidad, 0 = no la tiene).')
print('→ Este formato ya puede usarse directamente con cualquier modelo de scikit-learn.')

### 📌 Ejemplo 3 — Datos no estructurados (texto libre)

In [ ]:
# Los datos no estructurados no tienen una organización tabular
registros_log = [
    '[2026-06-01 08:32:11] INFO  Entrenamiento completado. Precisión: 94.3%',
    '[2026-06-01 08:35:02] ERROR No se encontró el archivo de datos: datos_test.csv',
    '[2026-06-01 08:36:50] WARN  El conjunto de datos tiene un 12% de valores nulos',
    '[2026-06-01 08:40:17] INFO  Modelo guardado en /models/v2.pkl'
]

print('Datos no estructurados — registros de log (texto libre):')
for registro in registros_log:
    print(' ', registro)

print('\n→ Para extraer información útil necesitaríamos técnicas como:')
print('   expresiones regulares, NLP, tokenización...')

### 📌 Ejemplo 3b — Estructurar texto libre con un modelo de lenguaje (LLM)

Las opiniones que escriben los clientes sobre un producto son **datos no estructurados**: texto libre, cada una con su longitud, su tono y sus giros. Para entrenar un modelo (o simplemente para contarlas) necesitamos convertirlas en algo **estructurado**, por ejemplo una columna categórica `sentimiento` con tres valores posibles: `POSITIVO`, `NEGATIVO` o `NEUTRAL`.

Una forma rápida de hacerlo es pedírselo a un **LLM** a través de una API. Usaremos [OpenRouter](https://openrouter.ai), que da acceso a muchos modelos (algunos gratuitos, marcados con `:free`) con la misma librería `openai` que se usa para ChatGPT.

> 🔑 **Antes de ejecutar:** crea una clave gratuita en <https://openrouter.ai/keys> y guárdala en un archivo llamado **`.env`** en la raíz del repositorio `DLCD`, con esta línea:
>
> ```
> OPENROUTER_API_KEY=sk-or-v1-...
> ```
>
> El archivo `.env` está en `.gitignore`, así que **nunca se sube a GitHub**. Si no existe, el cuaderno te pedirá la clave al ejecutar la celda.
>
> ⚠️ Nunca escribas una clave directamente en el código de un cuaderno: cualquiera que lo vea podría usarla.

In [ ]:
# DataFrame con opiniones de clientes sobre un producto (datos no estructurados)
opiniones = pd.DataFrame({
    'cliente': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008'],
    'opinion': [
        'Llegó en dos días y funciona perfectamente. Muy contento con la compra.',
        'La batería no dura ni medio día. Una decepción total, lo voy a devolver.',
        'Es un auricular normal, cumple con lo que promete y poco más.',
        'Calidad de sonido espectacular, lo recomiendo sin dudarlo.',
        'Tardó tres semanas en llegar y la caja venía abollada.',
        'El producto está bien, pero el servicio de atención al cliente fue pésimo.',
        'Lo uso para ir al gimnasio. Ni fu ni fa.',
        '¡Genial! Justo lo que buscaba y a buen precio.',
    ],
})
opiniones

In [ ]:
# Configuramos el cliente de OpenRouter (usa la librería openai)
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # lee OPENROUTER_API_KEY del archivo .env, si existe
api_key = os.environ.get('OPENROUTER_API_KEY') or getpass('Pega tu clave de OpenRouter: ')

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=api_key,
)

# Modelo gratuito de OpenRouter. Puedes cambiarlo por cualquier otro de https://openrouter.ai/models
MODELO = 'nvidia/nemotron-3-ultra-550b-a55b:free'
print('✅ Cliente de OpenRouter configurado')

In [ ]:
ETIQUETAS = ['POSITIVO', 'NEGATIVO', 'NEUTRAL']

INSTRUCCIONES = (
    'Eres un clasificador de sentimiento de opiniones de clientes. '
    'Responde ÚNICAMENTE con una de estas palabras: POSITIVO, NEGATIVO o NEUTRAL. '
    'Sin explicaciones ni signos de puntuación.'
)


def clasificar_sentimiento(texto):
    """Pide al LLM que clasifique una opinión y devuelve una de las ETIQUETAS."""
    try:
        respuesta = client.chat.completions.create(
            model=MODELO,
            messages=[
                {'role': 'system', 'content': INSTRUCCIONES},
                {'role': 'user', 'content': texto},
            ],
            temperature=0,  # respuestas lo más deterministas posible
        )
        contenido = (respuesta.choices[0].message.content or '').upper()
    except Exception as e:
        print(f'⚠️ Error con la opinión «{texto[:30]}...»: {e}')
        return 'ERROR'

    # El modelo puede añadir texto de más: nos quedamos con la etiqueta que aparezca
    for etiqueta in ETIQUETAS:
        if etiqueta in contenido:
            return etiqueta
    return 'DESCONOCIDO'


# Aplicamos la función a cada opinión → nueva columna estructurada
opiniones['sentimiento'] = opiniones['opinion'].apply(clasificar_sentimiento)
opiniones

In [ ]:
# Ahora el texto libre se ha convertido en una variable categórica que podemos analizar
print(opiniones['sentimiento'].value_counts())

> 💡 **Lo que acabamos de hacer:** a partir de una columna de **texto no estructurado** hemos generado una columna **categórica nominal** (`sentimiento`) que ya podemos contar, filtrar, graficar o usar como etiqueta para entrenar un modelo propio.
>
> Pero ojo:
> - El LLM **puede equivocarse**, sobre todo con opiniones mixtas (C006) o ambiguas (C007). Hay que revisar una muestra a mano.
> - Los modelos gratuitos tienen **límites de peticiones por minuto**: para miles de filas habría que agrupar opiniones o usar un modelo de pago.
> - Enviamos los textos a un servicio externo: **nunca** mandes datos personales o confidenciales.

---
### 🔧 Ejercicio 1 — Identifica la naturaleza de los datos

Para cada una de las siguientes fuentes de datos, indica si son **estructurados**, **semiestructurados** o **no estructurados**, y justifica brevemente tu respuesta.

In [ ]:
# Completa el diccionario con tus respuestas.
# Valores posibles: 'estructurado', 'semiestructurado', 'no estructurado'

fuentes = [
    'Tabla de una base de datos SQL con historial de ventas',
    'Imagen de una radiografía pulmonar',
    'Archivo CSV con temperaturas horarias de una ciudad',
    'Respuesta de una API REST con datos de un pedido en formato JSON',
    'Grabación de audio de una llamada al servicio de atención al cliente',
    'Archivo XML con configuración de un sistema SCADA',
    'Hoja de cálculo Excel con resultados de sensores IoT'
]

respuestas = {
    fuentes[0]: 'estructurado',  # Escribe tu respuesta
    fuentes[1]: 'no estructurado',
    fuentes[2]: 'estructurado',
    fuentes[3]: 'estructurado',
    fuentes[4]: 'no estructurado',
    fuentes[5]: 'estructurado',
    fuentes[6]: 'estructurado',
}

# Verificación
for fuente, respuesta in respuestas.items():
    print(f'{respuesta:20s} → {fuente}')

---
## 1.2 Tipos de variables

Dentro de un conjunto de datos estructurado, cada columna es una **variable**. Conocer el tipo de cada variable es fundamental antes de cualquier análisis o limpieza.

```
Variables
├── Numéricas
│   ├── Continuas   → pueden tomar cualquier valor real    (peso, temperatura, precio)
│   └── Discretas   → valores enteros contables            (número de hijos, unidades vendidas)
├── Categóricas
│   ├── Nominales   → sin orden entre categorías           (color, género)
│   └── Ordinales   → con orden definido entre categorías  (nivel educativo, satisfacción 1-5)
├── Temporales     → fechas y horas                       (fecha_nacimiento, timestamp)
└── Booleanas       → solo dos valores: True/False         (activo, tiene_seguro)
```

> ⚠️ **Importante:** pandas asigna tipos automáticamente (`int64`, `float64`, `object`, `bool`, `datetime64`), pero no siempre coinciden con el tipo estadístico real de la variable. Siempre hay que revisarlos.

### 📌 Ejemplo 4 — Tipos de variables en el dataset Titanic

In [ ]:
# Cargamos el dataset Titanic (incluido en seaborn)
titanic = sns.load_dataset('titanic')

print('Dataset: Titanic')
print(f'Dimensiones: {titanic.shape[0]} filas × {titanic.shape[1]} columnas\n')
titanic.head()

In [ ]:
# Tipos que asigna pandas automáticamente
print('Tipos de datos según pandas (dtypes):')
print(titanic.dtypes)

print('\n--- Clasificación estadística correcta ---')
clasificacion = {
    'survived':   'Booleana (0=no, 1=sí)',
    'pclass':     'Categórica ordinal (1º > 2º > 3º clase)',
    'sex':        'Categórica nominal',
    'age':        'Numérica continua',
    'sibsp':      'Numérica discreta (nº hermanos/cónyuge a bordo)',
    'parch':      'Numérica discreta (nº padres/hijos a bordo)',
    'fare':       'Numérica continua (precio del billete)',
    'embarked':   'Categórica nominal (puerto de embarque)',
    'who':        'Categórica nominal (man/woman/child)',
    'alone':      'Booleana'
}

for col, tipo in clasificacion.items():
    dtype_pandas = str(titanic[col].dtype)
    print(f'  {col:12s} → pandas: {dtype_pandas:10s} | estadístico: {tipo}')

### 📌 Ejemplo 5 — Cuando pandas se equivoca con el tipo

In [ ]:
# Creamos un dataset con errores típicos de tipado
datos_erroneos = pd.DataFrame({
    'edad':          ['23', '35', '28', '42'],         # Debería ser int, está como str
    'precio':        [12.5, 8.0, '15,3', 9.99],        # Coma decimal en vez de punto
    'fecha_compra':  ['2026-01-15', '2026-02-03',
                      '2026-03-20', '2026-01-28'],     # Fecha como texto
    'es_premium':    [1, 0, 1, 1],                     # Booleano codificado como int
    'categoria':     [1, 2, 1, 3]                      # Categórica codificada como int
})

print('Tipos detectados por pandas (antes de corregir):')
print(datos_erroneos.dtypes)
print()
print(datos_erroneos)
print('\n⚠️ Todos estos tipos deberían ser revisados y corregidos antes del análisis.')

---
### 🔧 Ejercicio 2 — Clasifica las variables del dataset Iris

Carga el dataset Iris (disponible en seaborn) y completa las tareas:

In [ ]:
# TAREA 1: Carga el dataset 'iris' con seaborn y muestra las primeras 5 filas
iris = sns.load_dataset('iris')

print(iris.head())

# TAREA 2: Muestra las dimensiones del dataset (filas × columnas)
print(f'Dimensiones: ...')

# TAREA 3: Muestra los tipos de datos que asigna pandas
print(iris.dtypes)

# TAREA 4: Para cada columna, indica su tipo estadístico real:
#   sepal_length → ???
#   sepal_width  → ???
#   petal_length → ???
#   petal_width  → ???
#   species      → ???

# TAREA 5: ¿Algún tipo detectado por pandas no coincide con el tipo estadístico?
# Escribe tu respuesta como comentario:
# Respuesta: ...
iris['species'] = iris['species'].astype('category')
print(iris.dtypes)
print(iris.head())


---
### 🔧 Ejercicio 3 — Corrige los tipos del dataset con errores

In [ ]:
# Partimos del DataFrame 'datos_erroneos' creado en el Ejemplo 5.
# Corrige los tipos de cada columna para que sean los adecuados.
# Pista: usa pd.to_numeric(), pd.to_datetime(), astype()

df_corregido = datos_erroneos.copy()

# 1. Corrige la columna 'edad' → debe ser int
# df_corregido['edad'] = ...
# 2. Corrige la columna 'precio' → debe ser float (ojo con la coma decimal)
# df_corregido['precio'] = ...

# 3. Corrige 'fecha_compra' → debe ser datetime
# df_corregido['fecha_compra'] = ...

# 4. Corrige 'es_premium' → debe ser bool
# df_corregido['es_premium'] = ...

# 5. Corrige 'categoria' → debe ser categórica
# df_corregido['categoria'] = ...

# Verifica el resultado
# print(df_corregido.dtypes)
# print(df_corregido)

---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| Datos estructurados | Tablas con filas y columnas. Formato nativo de pandas. |
| Datos semiestructurados | JSON, XML. Tienen organización pero no son directamente tabulares. |
| Datos no estructurados | Texto, imágenes, audio. Requieren técnicas especiales para ML. |
| Estructurar texto con un LLM | Un modelo de lenguaje puede convertir texto libre en una variable categórica (p. ej. sentimiento). |
| Tipo estadístico vs tipo pandas | No siempre coinciden. Siempre revisar con `dtypes`. |
| Variables numéricas | Continuas (reales) o discretas (enteras contables). |
| Variables categóricas | Nominales (sin orden) u ordinales (con orden). |

### 🔜 Próxima sesión
**Bloque 1 — Sesión 2:** Identificación de entidades, atributos y relaciones entre conjuntos de datos.